In [1]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached pyyaml-6.0.3-cp314-cp314-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached attrs-26.1.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached frozenlist-1.8.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (20 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
  Using cached sniffio-1.3.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached zstandard-0.25.0-cp314-cp314-macosx_11_0_arm64.whl.metadata (3.3 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
  Using cached packaging-24.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
  Using cached importlib_resourc

In [2]:
import os
from getpass import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LangSmith API Key: ")

In [3]:
print("LangSmith configured:", bool(os.getenv("LANGCHAIN_API_KEY")))

LangSmith configured: True


In [4]:
%pip install -U langchain-ollama langchain-chroma beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [langchain-chroma]

[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

response = llm.invoke("What is Retrieval-Augmented Generation?")

print(response.content)

Retrieval-Augmented Generation (RAG) is a type of artificial intelligence (AI) model that combines the strengths of retrieval-based models and generation-based models. It is a hybrid approach that leverages the capabilities of both to generate high-quality text.

In traditional generation models, the model generates text from scratch, using a learned representation of the input data. However, this approach can be limited by the model's ability to generate coherent and contextually relevant text.

Retrieval-Augmented Generation models, on the other hand, use a retrieval-based approach to first retrieve relevant information from a large database or knowledge graph. The retrieved information is then used as input to a generation model, which generates text based on the retrieved information.

The RAG model typically consists of two main components:

1. **Retrieval component**: This component is responsible for retrieving relevant information from a large database or knowledge graph. The r

In [6]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vector = embeddings.embed_query("What is RAG?")

print("Embedding dimensions:", len(vector))
print("First 5 values:", vector[:5])

Embedding dimensions: 768
First 5 values: [0.058354974, 0.009270313, -0.17480668, -0.1246053, 0.009553315]


In [1]:
import bs4

from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

/var/folders/14/4x_r7nkn58x6l26wq2qhg_pr0000gn/T/ipykernel_9157/3688525381.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

docs = loader.load()

print("Documents loaded:", len(docs))
print(docs[0].page_content[:500])

Documents loaded: 1


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In


In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

splits = text_splitter.split_documents(docs)

print("Number of chunks:", len(splits))
print(splits[0].page_content)

Number of chunks: 63
LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:

Planning

Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.


Memory


In [4]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 3}
)

print("Vector database created!")

Vector database created!


In [5]:
question = "What is Task Decomposition?"

retrieved_docs = retriever.invoke(question)

for i, doc in enumerate(retrieved_docs):
    print(f"\nDocument {i+1}")
    print(doc.page_content[:500])


Document 1
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an 

Document 2
Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.
Another quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermedi

In [6]:
template = """Answer the question based only on the following context.
If the answer is not in the context, say you don't know.

Context:
{context}

Question: {question}

Answer:
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

In [7]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [8]:
response = rag_chain.invoke(
    "What is Task Decomposition?"
)

print(response)

Task Decomposition is a process where a complicated task is broken down into smaller and simpler steps.


In [9]:
questions = [
    "What is the difference between short-term and long-term memory in AI agents?",
    "What are the challenges of using LLMs as autonomous agents?",
    "What is self-reflection in LLM agents?"
]

for question in questions:
    print("\nQuestion:", question)
    print("Answer:", rag_chain.invoke(question))


Question: What is the difference between short-term and long-term memory in AI agents?
Answer: According to the context, the difference between short-term and long-term memory in AI agents is as follows:

- Short-term memory (STM) is used for in-context learning and has a limited capacity (about 7 items) that lasts for a short duration (20-30 seconds).
- Long-term memory (LTM) is used for retaining and recalling information over extended periods, often by leveraging an external vector store and fast retrieval, with essentially unlimited storage capacity.

Question: What are the challenges of using LLMs as autonomous agents?
Answer: I don't know. The question "What are the challenges of using LLMs as autonomous agents?" is not present in the provided context.

Question: What is self-reflection in LLM agents?
Answer: Self-reflection in LLM agents is the ability to refine past action decisions, correct previous mistakes, and learn from experiences, allowing the agent to improve iterative

## Indexing

In [10]:
# Documents
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

In [11]:
import tiktoken

def num_tokens_from_string(string: str, encoding_name: str) -> int:
    """Returns the number of tokens in a text string."""
    encoding = tiktoken.get_encoding(encoding_name)
    num_tokens = len(encoding.encode(string))
    return num_tokens

print("Question tokens:", num_tokens_from_string(question, "cl100k_base"))
print("Document tokens:", num_tokens_from_string(document, "cl100k_base"))

Question tokens: 8
Document tokens: 7


In [13]:
from langchain_ollama import OllamaEmbeddings

embd = OllamaEmbeddings(model="nomic-embed-text")

query_result = embd.embed_query(question)
document_result = embd.embed_query(document)

print("Question embedding dimensions:", len(query_result))
print("Document embedding dimensions:", len(document_result))

Question embedding dimensions: 768
Document embedding dimensions: 768


In [14]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)

print("Cosine Similarity:", similarity)

Cosine Similarity: 0.7488087562682632


In [15]:
import bs4
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)

blog_docs = loader.load()

print("Documents loaded:", len(blog_docs))

Documents loaded: 1


In [16]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,
    chunk_overlap=50
)

indexing_splits = text_splitter.split_documents(blog_docs)

print("Number of chunks:", len(indexing_splits))
print("First chunk:", indexing_splits[0].page_content)

Number of chunks: 50
First chunk: LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, essays and programs; it can be framed as a powerful general problem solver.
Agent System Overview#
In a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:

Planning

Subgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.
Reflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.


Mem

In [17]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embd = OllamaEmbeddings(model="nomic-embed-text")

indexing_vectorstore = Chroma.from_documents(
    documents=indexing_splits,
    embedding=embd
)

indexing_retriever = indexing_vectorstore.as_retriever()

print("Indexing completed successfully!")

Indexing completed successfully!


## Retrieval

In [19]:
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

embd = OllamaEmbeddings(model="nomic-embed-text")

vectorstore = Chroma.from_documents(
    documents=indexing_splits,
    embedding=embd
)

retriever = vectorstore.as_retriever()

print("Retriever created successfully!")

Retriever created successfully!


In [21]:
retriever = indexing_vectorstore.as_retriever()

In [24]:
docs = retriever.invoke("What is Task Decomposition?")

for i, doc in enumerate(docs):
    print(f"\nDocument {i+1}")
    print(doc.page_content[:500])


Document 1
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks and shed lights into an 

Document 2
Component One: Planning#
A complicated task usually involves many steps. An agent needs to know what they are and plan ahead.
Task Decomposition#
Chain of thought (CoT; Wei et al. 2022) has become a standard prompting technique for enhancing model performance on complex tasks. The model is instructed to “think step by step” to utilize more test-time computation to decompose hard tasks into smaller and simpler steps. CoT transforms big tasks into multiple manageable tasks

In [25]:
len(docs)

4

In [26]:
for k in [1, 3, 5]:
    test_retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )

    results = test_retriever.invoke(
        "What is Task Decomposition?"
    )

    print(f"Requested {k} chunks, retrieved {len(results)}")

Requested 1 chunks, retrieved 1
Requested 3 chunks, retrieved 3
Requested 5 chunks, retrieved 5


## Generation

In [48]:
from langchain_core.prompts import ChatPromptTemplate

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

print(prompt)

input_variables=['context', 'question'] input_types={} partial_variables={} messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})]


In [49]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2",
    temperature=0
)

print("Llama 3.2 initialized!")

Llama 3.2 initialized!


In [50]:
chain = prompt | llm

print("Generation chain created!")

Generation chain created!


In [ ]:
response = chain.invoke({
    "context": "\n\n".join(doc.page_content for doc in docs),
    "question": "What is Task Decomposition?"
})

print(response.content)

Task Decomposition is the process of breaking down a complicated task into smaller, simpler steps.


Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a1ccfd-27be-7402-857a-95ca931a5911,id=01a1ccfd-27be-7402-857a-95ca931a5911; trace=01a1ccfd-27be-7402-857a-95ca931a5911,id=01a1ccfd-27bf-723a-9320-f53ca8b140bd; trace=01a1ccfd-27be-7402-857a-95ca931a5911,id=01a1ccfd-27bf-723a-9320-f53ca8b140bd; trace=01a1ccfd-27be-7402-857a-95ca931a5911,id=01a1ccfd-27c0-7623-b05b-0720faa5e073
Failed to send compressed multipart ingest: langsmith.utils.LangSmithAuthError: Authentication failed for https://api.smith.langchain.com/runs/multipart. HTTPError('401 Client Error: Unauthorized for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Unauthorized"}\n')trace=01a1ccfd-27be-7402-857a-95ca931a5911,id=01a1ccfd-27c0-7623-b05b-0720faa5e073; trace=01a1ccfd-

In [1]:
from langchain import hub
prompt_hub_rag = hub.pull("rlm/rag-prompt")

ImportError: cannot import name 'hub' from 'langchain' (/Users/aish/Desktop/rag_from_scratch/.venv/lib/python3.14/site-packages/langchain/__init__.py)

In [ ]:
prompt_hub_rag

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")